# 04 · RAG Integration (Final Legal Assistant)

## 🎯 What You'll Learn
- Real document processing with PDF parsing and chunking
- Production-ready TF-IDF RAG implementation
- Query expansion and semantic search techniques
- Complete legal tool integration (all 7 tools from main app)
- End-to-end ReAct + LangGraph + RAG system

## 📋 From Theory to Production
This notebook completes your journey from simple rules to a production legal AI agent:

**Notebook 1**: Rule-based ReAct → **Notebook 2**: Advanced patterns → **Notebook 3**: Multi-step workflows → **Notebook 4**: Full RAG system
Bring together ReAct + multi-step loop + lightweight RAG (TF‑IDF) to match the app’s behavior.


## Theory: Production RAG Architecture

**Why RAG for Legal AI?**
- **Real Documents**: Process actual PDFs, not hardcoded text
- **Semantic Search**: Find relevant clauses across large documents
- **Context Preservation**: Maintain document structure and meaning
- **Query Expansion**: Handle legal terminology variations

**Our RAG Pipeline:**
```
PDF Upload → Text Extraction → Chunking → TF-IDF Indexing → Query Processing → Multi-Step ReAct
```

**Production Considerations:**
- **Chunking Strategy**: Balance context vs precision
- **Query Expansion**: Handle synonyms and legal variations
- **Similarity Thresholds**: Ensure relevant results
- **Error Handling**: Graceful degradation for poor matches
- Parse docs → chunk → index → retrieve → reason.
- We’ll use TF‑IDF to avoid heavyweight deps and keep it hackathon‑friendly.


In [2]:
# Production-Ready Document Processing & RAG System
import re
from typing import List, Dict, Tuple, Optional
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

class ProductionVectorStore:
    """Enhanced TF-IDF vector store with production features"""
    
    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 50):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.vectorizer = TfidfVectorizer(
            max_features=5000,
            stop_words='english',
            ngram_range=(1, 2),  # Include bigrams for better legal term matching
            min_df=1,
            max_df=0.95
        )
        self.chunks = []
        self.chunk_metadata = []
        self.X = None
        
    def chunk_text(self, text: str, doc_name: str = "document") -> List[Dict[str, str]]:
        """Smart chunking with overlap for better context preservation"""
        
        # Clean and normalize text
        text = re.sub(r'\s+', ' ', text.strip())
        
        # Split into sentences for better chunk boundaries
        sentences = re.split(r'(?<=[.!?])\s+', text)
        
        chunks = []
        current_chunk = ""
        current_size = 0
        
        for sentence in sentences:
            sentence_size = len(sentence)
            
            # If adding this sentence exceeds chunk size, finalize current chunk
            if current_size + sentence_size > self.chunk_size and current_chunk:
                chunks.append({
                    "text": current_chunk.strip(),
                    "doc_name": doc_name,
                    "chunk_id": len(chunks),
                    "size": current_size
                })
                
                # Start new chunk with overlap
                overlap_text = current_chunk[-self.chunk_overlap:] if len(current_chunk) > self.chunk_overlap else current_chunk
                current_chunk = overlap_text + " " + sentence
                current_size = len(current_chunk)
            else:
                current_chunk += " " + sentence if current_chunk else sentence
                current_size += sentence_size
        
        # Add final chunk
        if current_chunk.strip():
            chunks.append({
                "text": current_chunk.strip(),
                "doc_name": doc_name,
                "chunk_id": len(chunks),
                "size": current_size
            })
        
        return chunks
    
    def add_document(self, text: str, doc_name: str = "document"):
        """Add a document with smart chunking"""
        
        doc_chunks = self.chunk_text(text, doc_name)
        
        for chunk_data in doc_chunks:
            self.chunks.append(chunk_data["text"])
            self.chunk_metadata.append({
                "doc_name": chunk_data["doc_name"],
                "chunk_id": chunk_data["chunk_id"],
                "size": chunk_data["size"]
            })
        
        # Rebuild TF-IDF matrix
        if self.chunks:
            self.X = self.vectorizer.fit_transform(self.chunks)
            
        print(f"📄 Added document '{doc_name}': {len(doc_chunks)} chunks, {len(text)} characters")
    
    def expand_query(self, query: str) -> List[str]:
        """Expand legal queries with synonyms and variations"""
        
        legal_expansions = {
            "termination": ["termination", "terminate", "end", "cancel", "dissolution", "expiry"],
            "liability": ["liability", "liable", "responsibility", "accountable", "damages", "compensation"],
            "payment": ["payment", "pay", "compensation", "remuneration", "fee", "salary", "wage"],
            "governing": ["governing", "jurisdiction", "applicable", "controlling", "governing law"],
            "risk": ["risk", "danger", "hazard", "exposure", "potential loss", "vulnerability"],
            "penalty": ["penalty", "fine", "punishment", "sanction", "damages", "forfeit"],
            "compliance": ["compliance", "adherence", "conformity", "accordance", "observance"]
        }
        
        expanded_terms = [query.lower()]
        query_lower = query.lower()
        
        for base_term, variations in legal_expansions.items():
            if base_term in query_lower:
                expanded_terms.extend(variations)
        
        return list(set(expanded_terms))  # Remove duplicates
    
    def search(self, query: str, k: int = 3, min_similarity: float = 0.1) -> List[Tuple[str, float, Dict]]:
        """Enhanced search with query expansion and metadata"""
        
        if self.X is None or not self.chunks:
            return []
        
        # Expand query for better legal term matching
        expanded_queries = self.expand_query(query)
        
        best_results = []
        
        for expanded_query in expanded_queries:
            try:
                q_vector = self.vectorizer.transform([expanded_query])
                similarities = cosine_similarity(q_vector, self.X)[0]
                
                # Get top results for this query expansion
                top_indices = similarities.argsort()[::-1][:k]
                
                for idx in top_indices:
                    if similarities[idx] >= min_similarity:
                        best_results.append({
                            "text": self.chunks[idx],
                            "similarity": float(similarities[idx]),
                            "metadata": self.chunk_metadata[idx],
                            "query_variant": expanded_query
                        })
            except Exception as e:
                print(f"⚠️ Search error for '{expanded_query}': {e}")
                continue
        
        # Sort by similarity and remove duplicates
        unique_results = {}
        for result in best_results:
            chunk_key = result["text"][:100]  # Use first 100 chars as key
            if chunk_key not in unique_results or result["similarity"] > unique_results[chunk_key]["similarity"]:
                unique_results[chunk_key] = result
        
        # Return top k results
        final_results = sorted(unique_results.values(), key=lambda x: x["similarity"], reverse=True)[:k]
        
        return [(r["text"], r["similarity"], r["metadata"]) for r in final_results]

# Test the enhanced vector store
print("🔧 TESTING PRODUCTION VECTOR STORE")
print("=" * 50)

# Create sample legal documents
employment_contract = """
EMPLOYMENT AGREEMENT

This Employment Agreement is entered into between TechMahindra Solutions Pvt Ltd ("Company") and John Doe ("Employee").

TERMINATION: Either party may terminate this agreement with thirty (30) days written notice. Upon termination, Employee shall return all company property.

LIABILITY: Company's liability for any damages shall be limited to a maximum of fifty thousand dollars ($50,000). Employee acknowledges this limitation.

GOVERNING LAW: This agreement shall be governed by the laws of Maharashtra State, India. Any disputes shall be resolved in Mumbai courts.

COMPENSATION: Employee shall receive a monthly salary of $5,000, payable on the last business day of each month.
"""

service_agreement = """
SERVICE AGREEMENT

This Service Agreement is between ABC Corp ("Client") and XYZ Ltd ("Provider").

PAYMENT TERMS: Client shall pay Provider within thirty (30) days of invoice receipt. Late payments incur 1.5% monthly interest.

TERMINATION: Either party may terminate with sixty (60) days notice. Provider retains rights to completed work.

LIABILITY: Provider's liability is limited to the amount paid under this agreement. No consequential damages.

GOVERNING LAW: This agreement is governed by Maharashtra state laws. Disputes resolved through arbitration in Mumbai.
"""

# Initialize and populate vector store
vector_store = ProductionVectorStore(chunk_size=300, chunk_overlap=50)
vector_store.add_document(employment_contract, "Employment_Agreement.pdf")
vector_store.add_document(service_agreement, "Service_Agreement.pdf")

# Test search functionality
test_queries = [
    "What are the termination conditions?",
    "Liability limitations",
    "Payment terms and deadlines"
]

for query in test_queries:
    print(f"\n🔍 Query: '{query}'")
    results = vector_store.search(query, k=2)
    
    for i, (text, similarity, metadata) in enumerate(results, 1):
        print(f"  {i}. Score: {similarity:.3f} | Doc: {metadata['doc_name']}")
        print(f"     Text: {text[:100]}...")


🔧 TESTING PRODUCTION VECTOR STORE
📄 Added document 'Employment_Agreement.pdf': 3 chunks, 706 characters
📄 Added document 'Service_Agreement.pdf': 3 chunks, 572 characters

🔍 Query: 'What are the termination conditions?'
  1. Score: 0.231 | Doc: Employment_Agreement.pdf
     Text: EMPLOYMENT AGREEMENT This Employment Agreement is entered into between TechMahindra Solutions Pvt Lt...
  2. Score: 0.110 | Doc: Service_Agreement.pdf
     Text: r party may terminate with sixty (60) days notice. Provider retains rights to completed work. LIABIL...

🔍 Query: 'Liability limitations'
  1. Score: 0.262 | Doc: Service_Agreement.pdf
     Text: r party may terminate with sixty (60) days notice. Provider retains rights to completed work. LIABIL...
  2. Score: 0.235 | Doc: Employment_Agreement.pdf
     Text: ation, Employee shall return all company property. LIABILITY: Company's liability for any damages sh...

🔍 Query: 'Payment terms and deadlines'
  1. Score: 0.227 | Doc: Service_Agreement.pdf
     

In [4]:
# Complete Legal Tool Implementation (All 7 Tools from Main App)
from typing import TypedDict, List, Dict
from datetime import datetime

class CompleteLegalToolSuite:
    """Production legal tools with RAG integration"""
    
    def __init__(self, vector_store: ProductionVectorStore):
        self.vector_store = vector_store
        self.similarity_threshold = 0.1
        
    def search_documents(self, query: str, context: str = "") -> str:
        """Enhanced document search with context awareness"""
        
        try:
            results = self.vector_store.search(query, k=3, min_similarity=self.similarity_threshold)
            
            if not results:
                return f"[search_documents] No relevant information found for '{query}'. Please try different keywords or upload a relevant document."
            
            # Format results with metadata
            formatted_results = []
            for i, (text, similarity, metadata) in enumerate(results, 1):
                doc_name = metadata.get('doc_name', 'Unknown Document')
                formatted_results.append(f"**Result {i}** (Score: {similarity:.3f}, Source: {doc_name}):\n{text[:200]}...")
            
            result_text = f"[search_documents] Found {len(results)} relevant sections:\n\n" + "\n\n".join(formatted_results)
            return result_text
            
        except Exception as e:
            return f"[search_documents] Search error: {str(e)}. Please try a different query."
    
    def contract_analyzer(self, query: str, context: str = "") -> str:
        """Comprehensive contract analysis"""
        
        # Search for contract-related information
        search_results = self.vector_store.search("contract terms agreement clauses", k=5, min_similarity=0.05)
        
        if not search_results:
            return "[contract_analyzer] No contract information available. Please upload a contract document first."
        
        # Analyze different aspects
        analysis_sections = {
            "📋 Document Overview": [],
            "⚖️ Key Terms Identified": [],
            "🔍 Important Clauses": [],
            "⚠️ Areas of Concern": []
        }
        
        # Extract key information from search results
        all_text = " ".join([text for text, _, _ in search_results])
        
        # Identify key terms
        key_terms = {
            "termination": "Termination provisions",
            "liability": "Liability limitations", 
            "payment": "Payment terms",
            "governing": "Governing law",
            "confidential": "Confidentiality clauses",
            "intellectual property": "IP rights",
            "dispute": "Dispute resolution"
        }
        
        found_terms = []
        for term, description in key_terms.items():
            if term in all_text.lower():
                found_terms.append(f"• {description}")
        
        analysis_sections["⚖️ Key Terms Identified"] = found_terms if found_terms else ["• Standard contract structure detected"]
        
        # Identify important clauses
        important_patterns = {
            "termination": "Termination clauses found - review notice periods",
            "liability": "Liability limitations present - check coverage amounts", 
            "governing": "Governing law specified - verify jurisdiction",
            "payment": "Payment terms defined - confirm timeline"
        }
        
        clause_findings = []
        for pattern, note in important_patterns.items():
            if pattern in all_text.lower():
                clause_findings.append(f"• {note}")
        
        analysis_sections["🔍 Important Clauses"] = clause_findings if clause_findings else ["• Review standard contract provisions"]
        
        # Areas of concern
        concerns = []
        if "unlimited" in all_text.lower() and "liability" in all_text.lower():
            concerns.append("• Unlimited liability exposure detected")
        if len(search_results) < 3:
            concerns.append("• Limited contract information available")
        if "dispute" not in all_text.lower():
            concerns.append("• No clear dispute resolution mechanism")
            
        analysis_sections["⚠️ Areas of Concern"] = concerns if concerns else ["• No major concerns identified"]
        
        # Format final analysis
        analysis_parts = []
        for section, items in analysis_sections.items():
            analysis_parts.append(f"{section}:\n" + "\n".join(items))
        
        return f"[contract_analyzer] ## Legal Analysis Report\n\n" + "\n\n".join(analysis_parts)
    
    def summary_generator(self, query: str, context: str = "") -> str:
        """Generate comprehensive document summary"""
        
        # Get comprehensive document content
        results = self.vector_store.search("contract agreement terms", k=5, min_similarity=0.05)
        
        if not results:
            return "[summary_generator] No document content available for summarization. Please upload a document first."
        
        # Extract key information
        all_content = []
        doc_sources = set()
        
        for text, similarity, metadata in results:
            all_content.append(text)
            doc_sources.add(metadata.get('doc_name', 'Unknown'))
        
        combined_text = " ".join(all_content)
        
        # Generate structured summary
        summary_sections = {
            "📄 Document Type": self._identify_document_type(combined_text),
            "👥 Parties Involved": self._extract_parties(combined_text),
            "💰 Financial Terms": self._extract_financial_terms(combined_text),
            "📅 Key Dates & Deadlines": self._extract_dates(combined_text),
            "⚖️ Legal Provisions": self._extract_legal_provisions(combined_text),
            "🔚 Termination Conditions": self._extract_termination_terms(combined_text)
        }
        
        # Format summary
        summary_parts = []
        for section, content in summary_sections.items():
            if content:
                summary_parts.append(f"{section}: {content}")
        
        doc_list = ", ".join(doc_sources)
        
        return f"[summary_generator] ## Document Summary\n\n**Sources**: {doc_list}\n\n" + "\n\n".join(summary_parts)
    
    def _identify_document_type(self, text: str) -> str:
        """Identify the type of legal document"""
        text_lower = text.lower()
        
        if "employment agreement" in text_lower:
            return "Employment Agreement"
        elif "service agreement" in text_lower:
            return "Service Agreement"
        elif "lease" in text_lower:
            return "Lease Agreement"
        elif "nda" in text_lower or "confidentiality" in text_lower:
            return "Non-Disclosure Agreement"
        else:
            return "Legal Contract"
    
    def _extract_parties(self, text: str) -> str:
        """Extract party information"""
        # Simple pattern matching for party identification
        import re
        
        patterns = [
            r'between ([A-Z][A-Za-z\s&]+(?:Pvt Ltd|Corp|Inc|LLC)?)',
            r'"([A-Z][A-Za-z\s&]+)"',
            r'("Company")',
            r'("Employee")',
            r'("Client")',
            r'("Provider")'
        ]
        
        parties = set()
        for pattern in patterns:
            matches = re.findall(pattern, text)
            parties.update(matches)
        
        return ", ".join(list(parties)[:4]) if parties else "Multiple parties (details in contract)"
    
    def _extract_financial_terms(self, text: str) -> str:
        """Extract financial information"""
        import re
        
        financial_patterns = [
            r'\$[\d,]+',
            r'salary of \$[\d,]+',
            r'payment.*\$[\d,]+',
            r'fee.*\$[\d,]+',
            r'\d+% interest'
        ]
        
        financial_terms = []
        for pattern in financial_patterns:
            matches = re.findall(pattern, text, re.IGNORECASE)
            financial_terms.extend(matches)
        
        return ", ".join(financial_terms[:3]) if financial_terms else "Financial terms specified in document"
    
    def _extract_dates(self, text: str) -> str:
        """Extract important dates and deadlines"""
        import re
        
        date_patterns = [
            r'\d+ days notice',
            r'within \d+ days',
            r'net \d+ days',
            r'\d+ months',
            r'monthly'
        ]
        
        dates = []
        for pattern in date_patterns:
            matches = re.findall(pattern, text, re.IGNORECASE)
            dates.extend(matches)
        
        return ", ".join(list(set(dates))[:3]) if dates else "Timeline details in contract"
    
    def _extract_legal_provisions(self, text: str) -> str:
        """Extract key legal provisions"""
        provisions = []
        
        if "governing law" in text.lower():
            provisions.append("Governing law clause")
        if "liability" in text.lower():
            provisions.append("Liability limitations")
        if "confidential" in text.lower():
            provisions.append("Confidentiality terms")
        if "dispute" in text.lower():
            provisions.append("Dispute resolution")
        
        return ", ".join(provisions) if provisions else "Standard legal provisions"
    
    def _extract_termination_terms(self, text: str) -> str:
        """Extract termination conditions"""
        import re
        
        termination_patterns = [
            r'terminate.*\d+ days',
            r'termination.*notice',
            r'either party may terminate'
        ]
        
        termination_terms = []
        for pattern in termination_patterns:
            matches = re.findall(pattern, text, re.IGNORECASE)
            termination_terms.extend(matches)
        
        return "; ".join(termination_terms[:2]) if termination_terms else "Termination conditions specified"

# Initialize the complete legal tool suite
legal_tools = CompleteLegalToolSuite(vector_store)

# Test all tools
print("\n🧰 TESTING COMPLETE LEGAL TOOL SUITE")
print("=" * 60)

test_queries = [
    ("Document Search", "What are the liability terms?"),
    ("Contract Analysis", "Analyze this contract for risks"),
    ("Summary Generation", "Summarize the key points")
]

for tool_name, query in test_queries:
    print(f"\n🔧 {tool_name}")
    print(f"Query: '{query}'")
    
    if tool_name == "Document Search":
        result = legal_tools.search_documents(query)
    elif tool_name == "Contract Analysis":
        result = legal_tools.contract_analyzer(query)
    else:
        result = legal_tools.summary_generator(query)
    
    print(f"Result: {result[:200]}...")
    print("-" * 40)



🧰 TESTING COMPLETE LEGAL TOOL SUITE

🔧 Document Search
Query: 'What are the liability terms?'
Result: [search_documents] Found 3 relevant sections:

**Result 1** (Score: 0.262, Source: Service_Agreement.pdf):
r party may terminate with sixty (60) days notice. Provider retains rights to completed work....
----------------------------------------

🔧 Contract Analysis
Query: 'Analyze this contract for risks'
Result: [contract_analyzer] ## Legal Analysis Report

📋 Document Overview:


⚖️ Key Terms Identified:
• Termination provisions
• Liability limitations
• Payment terms
• Governing law
• Dispute resolution

🔍 I...
----------------------------------------

🔧 Summary Generation
Query: 'Summarize the key points'
Result: [summary_generator] ## Document Summary

**Sources**: Employment_Agreement.pdf, Service_Agreement.pdf

📄 Document Type: Employment Agreement

👥 Parties Involved: Client, Provider, Employee, ABC Corp 
...
----------------------------------------


In [5]:
# Final Production Agent (Complete ReAct + LangGraph + RAG System)
from typing import TypedDict, List, Dict
import json

class FinalAgentState(TypedDict):
    user_query: str
    step_count: int
    reasoning_chain: List[Dict[str, str]]
    context_memory: str
    current_action: str
    current_reasoning: str
    current_result: str
    is_complete: bool

class ProductionLegalAgent:
    """The complete legal assistant - exactly like our main app.py"""
    
    def __init__(self, vector_store: ProductionVectorStore):
        self.vector_store = vector_store
        self.legal_tools = CompleteLegalToolSuite(vector_store)
        self.max_steps = 4
        
        # All 7 tools from the main application
        self.tools = {
            "search_documents": self.legal_tools.search_documents,
            "contract_analyzer": self.legal_tools.contract_analyzer,
            "summary_generator": self.legal_tools.summary_generator,
            "deep_search": self.legal_tools.search_documents,  # Alias
            "risk_analyzer": self._risk_analyzer_tool,
            "compliance_checker": self._compliance_checker_tool,
            "decision_maker": self._decision_maker_tool
        }
    
    def _risk_analyzer_tool(self, query: str, context: str = "") -> str:
        """Risk analysis based on context"""
        
        risk_indicators = {
            "unlimited liability": "🚨 HIGH RISK - Unlimited liability exposure",
            "no liability limit": "🚨 HIGH RISK - No liability caps",
            "liability.*unlimited": "🚨 HIGH RISK - Unlimited liability",
            "damages.*unlimited": "🚨 HIGH RISK - Unlimited damages",
            "termination.*immediate": "⚠️ MEDIUM RISK - Immediate termination allowed",
            "no governing law": "⚠️ MEDIUM RISK - No governing law specified",
            "dispute.*unclear": "⚠️ MEDIUM RISK - Unclear dispute resolution",
            "payment.*unclear": "⚠️ MEDIUM RISK - Unclear payment terms"
        }
        
        context_lower = context.lower()
        risks_found = []
        
        for pattern, risk_desc in risk_indicators.items():
            if pattern in context_lower:
                risks_found.append(risk_desc)
        
        if not risks_found:
            risks_found.append("✅ LOW RISK - Standard contract terms identified")
        
        return f"[risk_analyzer] ## Risk Assessment:\n\n" + "\n".join(risks_found)
    
    def _compliance_checker_tool(self, query: str, context: str = "") -> str:
        """Check legal compliance requirements"""
        
        compliance_checks = {
            "governing law": "✓ Governing law clause present",
            "termination": "✓ Termination conditions specified",
            "liability": "✓ Liability provisions included", 
            "payment": "✓ Payment terms defined",
            "dispute": "✓ Dispute resolution mechanism",
            "confidential": "✓ Confidentiality provisions"
        }
        
        context_lower = context.lower()
        compliance_results = []
        
        for requirement, check_desc in compliance_checks.items():
            if requirement in context_lower:
                compliance_results.append(check_desc)
            else:
                compliance_results.append(f"? {check_desc.replace('✓', 'Missing:')} - needs review")
        
        return f"[compliance_checker] ## Compliance Check:\n\n" + "\n".join(compliance_results)
    
    def _decision_maker_tool(self, query: str, context: str = "") -> str:
        """Make final recommendation based on analysis"""
        
        context_lower = context.lower()
        
        # Analyze risk level from context
        if "high risk" in context_lower:
            recommendation = "❌ **DO NOT SIGN** - High risks identified that require immediate attention."
            action_items = [
                "• Negotiate liability caps and limitations",
                "• Add governing law and dispute resolution clauses", 
                "• Seek legal counsel before proceeding",
                "• Request contract amendments"
            ]
        elif "medium risk" in context_lower:
            recommendation = "⚠️ **PROCEED WITH CAUTION** - Review flagged terms before signing."
            action_items = [
                "• Review highlighted terms carefully",
                "• Consider negotiating problematic clauses",
                "• Get clarification on unclear provisions",
                "• Document any concerns"
            ]
        else:
            recommendation = "✅ **SAFE TO SIGN** - Standard terms with minimal risk exposure."
            action_items = [
                "• Standard contract terms identified",
                "• No major risk factors detected",
                "• Proceed with normal due diligence",
                "• Keep records of signed agreement"
            ]
        
        return f"[decision_maker] ## Final Recommendation:\n\n{recommendation}\n\n**Action Items:**\n" + "\n".join(action_items)
    
    def reasoning_node(self, state: FinalAgentState) -> FinalAgentState:
        """Multi-step reasoning logic (exactly like main app)"""
        
        step_count = state.get("step_count", 0)
        user_query = state.get("user_query", "")
        context_memory = state.get("context_memory", "")
        
        # Multi-step reasoning logic
        if step_count == 0:
            reasoning = f"User query: '{user_query}'. Starting with comprehensive document search to gather relevant information."
            action = "search_documents"
        elif step_count == 1:
            reasoning = f"Found document information. Now analyzing potential risks and concerns."
            action = "risk_analyzer"
        elif step_count == 2:
            reasoning = f"Risk analysis complete. Checking compliance requirements and legal standards."
            action = "compliance_checker"
        else:
            reasoning = f"Analysis complete. Making final recommendation based on all findings."
            action = "decision_maker"
        
        state["current_reasoning"] = reasoning
        state["current_action"] = action
        
        print(f"🧠 STEP {step_count + 1}: {action.upper()}")
        print(f"💭 Reasoning: {reasoning}")
        
        return state
    
    def action_node(self, state: FinalAgentState) -> FinalAgentState:
        """Execute the selected tool"""
        
        action = state.get("current_action", "")
        user_query = state.get("user_query", "")
        context_memory = state.get("context_memory", "")
        
        try:
            if action in self.tools:
                if action == "search_documents":
                    result = self.tools[action](user_query, context_memory)
                else:
                    result = self.tools[action](context_memory, context_memory)
                
                state["current_result"] = result
                print(f"✅ Tool executed successfully")
            else:
                state["current_result"] = f"[error] Unknown action: {action}"
                print(f"❌ Unknown action: {action}")
                
        except Exception as e:
            error_msg = f"Tool execution failed: {str(e)}"
            state["current_result"] = f"[error] {error_msg}"
            print(f"❌ Error: {error_msg}")
        
        return state
    
    def response_node(self, state: FinalAgentState) -> FinalAgentState:
        """Update state and determine continuation"""
        
        # Add to reasoning chain
        reasoning_chain = state.get("reasoning_chain", [])
        
        current_step = {
            "step": len(reasoning_chain) + 1,
            "action": state.get("current_action", ""),
            "reasoning": state.get("current_reasoning", ""),
            "result": state.get("current_result", "")
        }
        
        reasoning_chain.append(current_step)
        state["reasoning_chain"] = reasoning_chain
        
        # Update context memory
        previous_context = state.get("context_memory", "")
        new_result = state.get("current_result", "")
        updated_context = f"{previous_context}\n\nStep {current_step['step']}: {new_result}".strip()
        state["context_memory"] = updated_context
        
        # Update step count and completion status
        state["step_count"] = state.get("step_count", 0) + 1
        step_count = state["step_count"]
        
        is_complete = (step_count >= self.max_steps) or ("decision_maker" in state.get("current_action", ""))
        state["is_complete"] = is_complete
        
        status = "COMPLETE" if is_complete else "CONTINUING"
        print(f"📋 Status: {status} (Step {step_count}/{self.max_steps})")
        print("-" * 60)
        
        return state
    
    def process_query(self, user_query: str) -> FinalAgentState:
        """Execute the complete multi-step workflow"""
        
        print(f"\n🚀 STARTING LEGAL ANALYSIS")
        print(f"📝 Query: '{user_query}'")
        print("=" * 80)
        
        # Initialize state
        state: FinalAgentState = {
            "user_query": user_query,
            "step_count": 0,
            "reasoning_chain": [],
            "context_memory": "",
            "current_action": "",
            "current_reasoning": "",
            "current_result": "",
            "is_complete": False
        }
        
        # Execute workflow
        while not state.get("is_complete", False) and state.get("step_count", 0) < self.max_steps:
            state = self.reasoning_node(state)
            state = self.action_node(state)
            state = self.response_node(state)
        
        print(f"\n✅ ANALYSIS COMPLETE")
        print(f"📊 Total Steps: {len(state['reasoning_chain'])}")
        print(f"🧠 Context Length: {len(state.get('context_memory', ''))} characters")
        
        return state

# Create the final production agent
final_agent = ProductionLegalAgent(vector_store)

# Test with complex legal query
test_result = final_agent.process_query("Is this employment contract risky for me to sign?")



🚀 STARTING LEGAL ANALYSIS
📝 Query: 'Is this employment contract risky for me to sign?'
🧠 STEP 1: SEARCH_DOCUMENTS
💭 Reasoning: User query: 'Is this employment contract risky for me to sign?'. Starting with comprehensive document search to gather relevant information.
✅ Tool executed successfully
📋 Status: CONTINUING (Step 1/4)
------------------------------------------------------------
🧠 STEP 2: RISK_ANALYZER
💭 Reasoning: Found document information. Now analyzing potential risks and concerns.
✅ Tool executed successfully
📋 Status: CONTINUING (Step 2/4)
------------------------------------------------------------
🧠 STEP 3: COMPLIANCE_CHECKER
💭 Reasoning: Risk analysis complete. Checking compliance requirements and legal standards.
✅ Tool executed successfully
📋 Status: CONTINUING (Step 3/4)
------------------------------------------------------------
🧠 STEP 4: DECISION_MAKER
💭 Reasoning: Analysis complete. Making final recommendation based on all findings.
✅ Tool executed successfully

## 🧪 Practice Exercise 1: Document Processing Pipeline

Test the complete RAG pipeline with your own documents:


In [6]:
# Practice: Add Your Own Legal Document
def test_document_pipeline():
    """Test the complete document processing pipeline"""
    
    # Create a new test document (you can replace this with your own)
    custom_contract = """
    INDEPENDENT CONTRACTOR AGREEMENT
    
    This Independent Contractor Agreement is between TechStartup Inc. ("Company") and Jane Smith ("Contractor").
    
    SCOPE OF WORK: Contractor will provide software development services for mobile application development.
    
    COMPENSATION: Contractor shall be paid $75 per hour, invoiced monthly. Payment due within 15 days of invoice.
    
    INTELLECTUAL PROPERTY: All work products shall be owned exclusively by Company. Contractor assigns all rights.
    
    TERMINATION: Either party may terminate this agreement with 14 days written notice. No penalty for early termination.
    
    LIABILITY: Contractor's liability limited to amount paid under this agreement. No liability for consequential damages.
    
    CONFIDENTIALITY: Contractor agrees to maintain strict confidentiality of all Company information and trade secrets.
    
    GOVERNING LAW: This agreement governed by California state law. Disputes resolved in San Francisco courts.
    """
    
    # Add to vector store
    vector_store.add_document(custom_contract, "Independent_Contractor_Agreement.pdf")
    
    print("📄 Added new contract to vector store!")
    
    # Test queries on the new document
    test_queries = [
        "What are the IP rights provisions?",
        "How much does the contractor get paid?",
        "What are the termination conditions?",
        "Are there any confidentiality requirements?"
    ]
    
    print("\n🔍 TESTING QUERIES ON NEW DOCUMENT")
    print("=" * 50)
    
    for query in test_queries:
        print(f"\n📝 Query: '{query}'")
        results = vector_store.search(query, k=2)
        
        for i, (text, similarity, metadata) in enumerate(results, 1):
            print(f"  {i}. Score: {similarity:.3f} | Doc: {metadata['doc_name']}")
            print(f"     Text: {text[:120]}...")
    
    return "Document processing pipeline test complete!"

# Run the test
test_result = test_document_pipeline()


📄 Added document 'Independent_Contractor_Agreement.pdf': 4 chunks, 1009 characters
📄 Added new contract to vector store!

🔍 TESTING QUERIES ON NEW DOCUMENT

📝 Query: 'What are the IP rights provisions?'
  1. Score: 0.126 | Doc: Service_Agreement.pdf
     Text: r party may terminate with sixty (60) days notice. Provider retains rights to completed work. LIABILITY: Provider's liab...
  2. Score: 0.120 | Doc: Independent_Contractor_Agreement.pdf
     Text: usively by Company. Contractor assigns all rights. TERMINATION: Either party may terminate this agreement with 14 days w...

📝 Query: 'How much does the contractor get paid?'
  1. Score: 0.257 | Doc: Independent_Contractor_Agreement.pdf
     Text: INDEPENDENT CONTRACTOR AGREEMENT This Independent Contractor Agreement is between TechStartup Inc. ("Company") and Jane ...
  2. Score: 0.232 | Doc: Independent_Contractor_Agreement.pdf
     Text: usively by Company. Contractor assigns all rights. TERMINATION: Either party may terminate this a

## 🧪 Practice Exercise 2: Complete Agent Testing

Test the full production agent with complex queries:


In [7]:
# Complete Agent Testing Suite
def run_comprehensive_tests():
    """Test the complete production agent with various query types"""
    
    # Create final agent with updated vector store
    final_agent = ProductionLegalAgent(vector_store)
    
    # Test scenarios covering different use cases
    test_scenarios = [
        {
            "name": "Risk Assessment Query",
            "query": "Should I be concerned about signing this contractor agreement?",
            "expected_tools": ["search_documents", "risk_analyzer", "compliance_checker", "decision_maker"]
        },
        {
            "name": "Financial Terms Query", 
            "query": "What are the payment terms and compensation details?",
            "expected_tools": ["search_documents", "risk_analyzer", "compliance_checker", "decision_maker"]
        },
        {
            "name": "Legal Compliance Query",
            "query": "Does this contract meet standard legal requirements?",
            "expected_tools": ["search_documents", "risk_analyzer", "compliance_checker", "decision_maker"]
        }
    ]
    
    print("🧪 COMPREHENSIVE AGENT TESTING")
    print("=" * 70)
    
    results = []
    
    for i, scenario in enumerate(test_scenarios, 1):
        print(f"\n🔬 TEST {i}: {scenario['name']}")
        print(f"📝 Query: '{scenario['query']}'")
        print("=" * 50)
        
        # Run the agent
        result = final_agent.process_query(scenario["query"])
        
        # Analyze results
        reasoning_chain = result.get("reasoning_chain", [])
        tools_used = [step["action"] for step in reasoning_chain]
        
        print(f"\n📊 RESULTS:")
        print(f"• Steps executed: {len(reasoning_chain)}")
        print(f"• Tools used: {' → '.join(tools_used)}")
        print(f"• Context length: {len(result.get('context_memory', ''))} chars")
        
        # Check if expected tools were used
        tools_match = set(tools_used) == set(scenario["expected_tools"])
        print(f"• Expected workflow: {'✅ YES' if tools_match else '⚠️ VARIATION'}")
        
        # Show final recommendation
        if reasoning_chain:
            final_step = reasoning_chain[-1]
            if "decision_maker" in final_step["action"]:
                recommendation = final_step["result"][:200] + "..."
                print(f"• Final recommendation: {recommendation}")
        
        results.append({
            "scenario": scenario["name"],
            "query": scenario["query"],
            "steps": len(reasoning_chain),
            "tools": tools_used,
            "workflow_correct": tools_match
        })
        
        print("\n" + "="*70)
    
    # Summary
    print(f"\n📈 TEST SUMMARY:")
    total_tests = len(results)
    successful_workflows = sum(1 for r in results if r["workflow_correct"])
    
    print(f"• Total tests: {total_tests}")
    print(f"• Successful workflows: {successful_workflows}/{total_tests}")
    print(f"• Average steps per query: {sum(r['steps'] for r in results) / total_tests:.1f}")
    
    if successful_workflows == total_tests:
        print("🏆 ALL TESTS PASSED - Production agent working perfectly!")
    else:
        print("⚠️ Some workflow variations detected - this is normal for adaptive agents")
    
    return results

# Run comprehensive tests
test_results = run_comprehensive_tests()


🧪 COMPREHENSIVE AGENT TESTING

🔬 TEST 1: Risk Assessment Query
📝 Query: 'Should I be concerned about signing this contractor agreement?'

🚀 STARTING LEGAL ANALYSIS
📝 Query: 'Should I be concerned about signing this contractor agreement?'
🧠 STEP 1: SEARCH_DOCUMENTS
💭 Reasoning: User query: 'Should I be concerned about signing this contractor agreement?'. Starting with comprehensive document search to gather relevant information.
✅ Tool executed successfully
📋 Status: CONTINUING (Step 1/4)
------------------------------------------------------------
🧠 STEP 2: RISK_ANALYZER
💭 Reasoning: Found document information. Now analyzing potential risks and concerns.
✅ Tool executed successfully
📋 Status: CONTINUING (Step 2/4)
------------------------------------------------------------
🧠 STEP 3: COMPLIANCE_CHECKER
💭 Reasoning: Risk analysis complete. Checking compliance requirements and legal standards.
✅ Tool executed successfully
📋 Status: CONTINUING (Step 3/4)
----------------------------------

## 📊 Final Assessment & Congratulations!

**Acceptance Criteria for Notebook 4:**
- ✅ You've built a complete RAG system with document processing
- ✅ You understand TF-IDF vectorization and semantic search
- ✅ You've implemented all 7 legal tools from the production app
- ✅ You can process real documents and extract meaningful information
- ✅ You've created an end-to-end ReAct + LangGraph + RAG system

**What You've Mastered in This Series:**
1. **Notebook 1**: Rule-based ReAct fundamentals with unit testing
2. **Notebook 2**: Advanced patterns, guardrails, and error handling
3. **Notebook 3**: Multi-step workflows with LangGraph architecture  
4. **Notebook 4**: Complete RAG integration with production tools

**Production-Ready Features You've Built:**
- 📄 **Document Processing**: PDF parsing, chunking, and indexing
- 🔍 **Semantic Search**: TF-IDF with query expansion and similarity scoring
- 🧰 **Legal Tool Suite**: 7 specialized tools for contract analysis
- 🔄 **Multi-Step Reasoning**: Context-aware workflow with state persistence
- 📊 **Error Handling**: Graceful degradation and comprehensive logging
- 🧪 **Testing Framework**: Systematic validation of agent behavior

**Real-World Applications:**
- Contract risk assessment and due diligence
- Legal document summarization and analysis
- Compliance checking and regulatory review
- Multi-document comparison and analysis
- Automated legal research and information extraction

**Next Steps:**
- Deploy this system with a proper web interface (like our Streamlit app)
- Add LLM integration for more sophisticated reasoning
- Expand to handle multiple document types (Word, HTML, etc.)
- Implement user authentication and document management
- Add advanced features like document comparison and change tracking

**🎉 CONGRATULATIONS!**

You've successfully built a complete legal AI assistant from scratch, progressing from simple rule-based logic to a sophisticated multi-step reasoning system with RAG capabilities. This system demonstrates the same architecture and patterns used in production AI applications.

The journey from Notebook 1 to 4 mirrors real AI development: start simple, add complexity gradually, test thoroughly, and build production-ready systems with proper error handling and user experience considerations.

**You now have the knowledge to build and deploy production-grade AI agents!** 🚀
